# NB64: E-commerce Fulfillment

Kafka -> Spark -> Redis/Mongo

## 1. Environment Setup

Installs **Java 8**, **Spark 3.5.0**, **Kafka 3.6.1**, and Python libraries (PySpark, Kafka-Python, Redis, Mongo, ES, Cassandra, MinIO).

In [ ]:
# Install Dependencies (Java 8, Spark 3.5.0, Kafka 3.6.1)
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!wget -q https://archive.apache.org/dist/kafka/3.6.1/kafka_2.13-3.6.1.tgz
!tar xf kafka_2.13-3.6.1.tgz
!pip uninstall -y numpy
!pip install -q "numpy<2.0.0"
!pip install -q findspark pyspark kafka-python redis pymongo elasticsearch==7.10.1 cassandra-driver minio

# Environment Variables
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"
import findspark
findspark.init()

## 2. Start Services

Starts background services needed for this pipeline:
- **Kafka** (Zookeeper + Broker)
- **Redis**
- **MongoDB**

In [ ]:
# Start Kafka
!./kafka_2.13-3.6.1/bin/zookeeper-server-start.sh -daemon ./kafka_2.13-3.6.1/config/zookeeper.properties
!./kafka_2.13-3.6.1/bin/kafka-server-start.sh -daemon ./kafka_2.13-3.6.1/config/server.properties
# Start Redis
!apt-get install redis-server -qq > /dev/null
!service redis-server start
# Start MongoDB
!wget -qO - https://www.mongodb.org/static/pgp/server-6.0.asc | apt-key add -
!echo "deb [ arch=amd64,arm64 ] https://repo.mongodb.org/apt/ubuntu jammy/mongodb-org/6.0 multiverse" | tee /etc/apt/sources.list.d/mongodb-org-6.0.list
!apt-get update -qq > /dev/null
!apt-get install -y mongodb-org -qq > /dev/null
!mkdir -p /data/db
!mongod --fork --logpath /var/log/mongodb.log --bind_ip 127.0.0.1

import time, socket, os
def wait_for_port(port, host='localhost', timeout=120):
    start_time = time.time()
    while True:
        try:
            with socket.create_connection((host, port), timeout=1):
                print(f"Service at {host}:{port} is ready!")
                return True
        except (OSError, ConnectionRefusedError):
            if time.time() - start_time > timeout:
                print(f"Timeout waiting for {host}:{port} to start.")
                # Dump logs for debugging
                if os.path.exists('minio.log'):
                    print('--- MINIO LOG ---')
                    print(open('minio.log').read())
                if os.path.exists('es.log'):
                    print('--- ES LOG ---')
                    print(open('es.log').read())
                if os.path.exists('cassandra.log'):
                    print('--- CASSANDRA LOG ---')
                    print(open('cassandra.log').read())
                raise Exception(f"Service at {host}:{port} failed to start.")
            time.sleep(2)

# Wait for services
wait_for_port(9092) # Kafka
wait_for_port(6379) # Redis
wait_for_port(27017) # MongoDB


## 3. Create Kafka Topic

Creates a topic named `input-topic`.

In [ ]:
# Create Topic
!./kafka_2.13-3.6.1/bin/kafka-topics.sh --create --topic input-topic --bootstrap-server localhost:9092 --replication-factor 1 --partitions 1

## 4. Producer (Orders)

Simulates order placement.

In [ ]:
from kafka import KafkaProducer
import json, time, random
print("Starting Order Producer...")
producer = KafkaProducer(bootstrap_servers='localhost:9092')
print("Sending 100 orders...")
for i in range(100):
    order = {'order_id': i, 'item': f'item_{random.randint(1,5)}', 'qty': 1}
    producer.send('input-topic', json.dumps(order).encode('utf-8'))
producer.flush()
print("Producer finished.")

## 5. Fulfillment Engine

1. Checks Redis Inventory.
2. Decrements if available.
3. Confirms order to MongoDB or Fails it.

In [ ]:
%%writefile kafka_consumer.py
from pyspark.sql import SparkSession
import redis, json
from pymongo import MongoClient

# Setup Inventory
r = redis.Redis()
for i in range(1,6): r.set(f"inv:item_{i}", 10)

spark = SparkSession.builder.appName("Ecommerce").getOrCreate()

def process_batch(df, epoch_id):
    rows = df.collect()
    r_local = redis.Redis()
    mongo = MongoClient()
    db = mongo.shop
    for row in rows:
        order = json.loads(row.value)
        item = order['item']
        # Decr Inventory
        new_qty = r_local.decr(f"inv:{item}")
        if new_qty >= 0:
            order['status'] = 'CONFIRMED'
            db.orders.insert_one(order)
        else:
            order['status'] = 'FAILED'
            db.failed_orders.insert_one(order)
    print(f"Batch {epoch_id} processed: Inventory updated & Orders recorded.")

print("Starting Spark Streaming Job...")
df = spark.readStream.format("kafka").option("kafka.bootstrap.servers", "localhost:9092").option("subscribe", "input-topic").option("startingOffsets", "earliest").load()
query = df.selectExpr("CAST(value AS STRING)").writeStream.foreachBatch(process_batch).start()
query.awaitTermination(30)
print("Spark Job Finished.")

In [ ]:
!spark-submit --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0 kafka_consumer.py

## 6. Verification

Check Confirmed vs Failed orders in Mongo.

In [ ]:
from pymongo import MongoClient
m = MongoClient()
print(f"Confirmed Orders: {m.shop.orders.count_documents({})}")
print(f"Failed Orders: {m.shop.failed_orders.count_documents({})}")